In [16]:
# Load libraries
import pyhocon
from tqdm import tqdm
from itertools import combinations
from sklearn.utils import shuffle
from torch.utils import data
from torch.utils.data import DataLoader

from data_loader import MultiTaskBatchSampler
from models import FullCrossEncoder
from evaluator import Evaluation
from dataset import CrossEncoderDatasetFull,CrossEncoderMixedDatasetFull, ParaphraseDataset
from utils import *


In [17]:
#Load config files
config = pyhocon.ConfigFactory.parse_file('configs/config_pairwise.json')
fix_seed(config)
logger = create_logger(config, create_file=True)
logger.info(pyhocon.HOCONConverter.convert(config, "hocon"))
create_folder(config.model_path)
print(config.experiment_id)

Experiment5


In [18]:
config_experiments = pyhocon.ConfigFactory.parse_file('configs/config_experiments.json')
datasets_config = pyhocon.ConfigFactory.parse_file('configs/config_datasets.json')
datasets_config = resolve_data_path(datasets_config)

### Load datasets according to config file

In [19]:
config.data_folder

'mrpc'

In [20]:
# init train and dev set
if "Experiment" in config.experiment_id:
    print("loads experiment", config.experiment_id)
    train = CrossEncoderMixedDatasetFull(config,datasets_config,config_experiments[config.experiment_id]['train'])
    val = CrossEncoderMixedDatasetFull(config,datasets_config,config_experiments[config.experiment_id]['val'])
    #test = CrossEncoderMixedDatasetFull(config,datasets_config,config_experiments[config.experiment_id]['test'])

else:
    if config.data_folder == "mrpc" or "HoW_DatasetDict" in config.data_folder:
        train = ParaphraseDataset(config, 'train') 
        val = ParaphraseDataset(config, 'validation')
    else:
        train = CrossEncoderDatasetFull(config, 'train') 
        val = CrossEncoderDatasetFull(config, 'val')

loads experiment Experiment5
-----------------
dataset_key ECBPLUS_ENTITY_CONFIG
split train
negative_sampling True
mention_type entity_mentions
num_mentions 6055
num_instances 209254
-----------------
dataset_key ECBPLUS_EVENT_CONFIG
split train
negative_sampling True
mention_type event_mentions
num_mentions 2892
num_instances 81189
-----------------
dataset_key FCC_T_ENTITY_CONFIG
split train
negative_sampling True
mention_type entity_mentions_attr
num_mentions 3975
num_instances 6625
-----------------
dataset_key FCC_T_EVENT_CONFIG
split train
negative_sampling True
mention_type event_mentions
topic 7CqELkejZUeV3JiDCpWSs6  has only one mention. Mentions pair not created for dataset  FCC-prep/event_mentions/train
topic QSTmTYGsbkA7nrrDBDVsA9  has only one mention. Mentions pair not created for dataset  FCC-prep/event_mentions/train
num_mentions 1602
num_instances 16073
-----------------
dataset_key MRPC
split train
negative_sampling False
num_instances 3668
-----------------
dataset_

In [21]:
#Debug sizes, splits


In [7]:
sampler = MultiTaskBatchSampler(        dataset=train,
        batch_size=32,
        mix_opt=1,
        extra_task_ratio=1,
        current_epoch=1,
        total_epochs=10,
        bin_size=64,
        bin_on=False,
        bin_grow_ratio=0.5,
        sampling = "sequential")
        
train_dataloader = DataLoader(
        train,
        batch_sampler=sampler,
        pin_memory=True
        )

In [22]:
val_loader = data.DataLoader(val, batch_size=config.batch_size, shuffle=False, pin_memory=True)

In [23]:
fcc_t_instances = ['FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]def1d8aa7/4/Jfwa', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]def1d8aa7/5/3wC5', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]def1d8aa7/6/dWT8', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]def1d8aa7/8/NWqU', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]de7a78e41/1/3nxa', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]de7a78e41/1/n8Tp', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]de7a78e41/2/GBAy', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]de7a78e41/2/Pdco', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]de7a78e41/5/Wues', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]de7a78e41/6/SGZh', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]de7a78e41/9/eZve', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]de7a78e41/9/o4vg', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/1/dUCh', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/2/PvnD', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/2/Kk3j', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/4/b3ao', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/5/BCaJ', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/6/7cFs', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/8/FeUJ', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/9/PKY5', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/11/WpgW', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/12/FXMb', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/13/DCDN', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/14/e88n', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/15/TpGS', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/17/eZNh', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/18/BmaN', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/20/MZJB', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/21/fWZ9', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/22/YCwN', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/23/P6pS', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/25/WxwY', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/27/HPaE', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/30/bjCJ', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/32/AQca', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/34/gAV5', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/35/gxT4', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/37/HFg6', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/42/PFs5', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/43/oHWu', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/46/6mTY', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/49/gXcR', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/50/Q6KE', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d1344cc2d/1/nGxc', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d1344cc2d/2/jaAS', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d1344cc2d/4/E6BP', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d1344cc2d/5/DnMy', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d1344cc2d/7/ZMSa', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d1344cc2d/9/Rnsx', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d1344cc2d/11/6kXy', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d1344cc2d/12/NWXN', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d95e3afb9/1/C2Xh', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d95e3afb9/2/eEkQ', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d95e3afb9/5/4afJ', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d95e3afb9/7/J2Vm', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d95e3afb9/9/iD3C', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d558f79c4/1/Nb9Q', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d558f79c4/2/FT3N', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d558f79c4/4/Frjc', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d558f79c4/5/i3we', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d558f79c4/6/3SDy', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d558f79c4/7/DtcB', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d558f79c4/9/9VAY', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d558f79c4/10/TctP', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d558f79c4/10/M8A3', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d558f79c4/12/dWJ4', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/1/Jcau', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/2/WYvK', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/3/XWuE', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/5/hrox', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/5/99fU', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/6/dBn8', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/7/ZwgD', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/7/ZkSg', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/8/66hT', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/8/AXz4', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/11/AaRq', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/12/EcHt', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/16/hrZ5', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/17/bjFi', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/18/TcRf', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/19/oFkG', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/21/8THw', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/22/aMF8', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/24/5meC', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/25/8L22', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d62f3c572/26/N6nM', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]da7069778/1/PPUs', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]da7069778/3/PV6S', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]da7069778/4/Etsg', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]da7069778/6/ANET', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d5442a2de/1/SFdY', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d5442a2de/2/ecJF', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d5442a2de/4/MrDf', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d5442a2de/5/egow', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d5c1372c3/1/cTDd', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d5c1372c3/2/k3V6', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d5c1372c3/3/bC2X', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d5c1372c3/5/Kz7y', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d5c1372c3/6/gspr', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d5c1372c3/9/Lynv', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d5c1372c3/14/VuHG', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]ddbab361c/13/bcqj', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]ddbab361c/17/SnfA', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]de7a78e41/10/nCnj', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]de7a78e41/10/aied', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/29/MwTi', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/36/CdzR', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/38/gUDg', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/40/cHWs', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/47/gTcT', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]dee8b2473/51/CDRZ', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d1344cc2d/8/kcwf', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d95e3afb9/4/EHDo', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d95e3afb9/8/TBrF', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d5c1372c3/8/atet', 'FCC-prep/entity_mentions_attr/val[SEP]dee8b2473/52/gLaF[SEP]d5c1372c3/12/KFJz', 'FCC-prep/entity_mentions_attr/val[SEP]d62f3c572/13/8r6x[SEP]d62f3c572/15/kNVB', 'FCC-prep/entity_mentions_attr/val[SEP]d62f3c572/13/8r6x[SEP]da7069778/2/LWNX', 'FCC-prep/entity_mentions_attr/val[SEP]d62f3c572/13/8r6x[SEP]d5c1372c3/7/fLjX', 'FCC-prep/entity_mentions_attr/val[SEP]d62f3c572/13/8r6x[SEP]d5c1372c3/11/KTNa', 'FCC-prep/entity_mentions_attr/val[SEP]d62f3c572/13/8r6x[SEP]ddbab361c/1/BGtB', 'FCC-prep/entity_mentions_attr/val[SEP]d62f3c572/13/8r6x[SEP]ddbab361c/2/bZX8', 'FCC-prep/entity_mentions_attr/val[SEP]d62f3c572/13/8r6x[SEP]ddbab361c/4/36kJ', 'FCC-prep/entity_mentions_attr/val[SEP]d62f3c572/13/8r6x[SEP]ddbab361c/5/hNVK', 'FCC-prep/entity_mentions_attr/val[SEP]d62f3c572/13/8r6x[SEP]ddbab361c/6/gL2o', 'FCC-prep/entity_mentions_attr/val[SEP]d62f3c572/13/8r6x[SEP]ddbab361c/7/fQUX', 'FCC-prep/entity_mentions_attr/val[SEP]d62f3c572/13/8r6x[SEP]ddbab361c/9/9EGQ']

In [26]:
sensitive_batches = []
for batch_x, batch_y,instance_id,dataset_task in val_loader:
    for batch,ins in zip(batch_x,instance_id):
        if ins in fcc_t_instances:
            sensitive_batches.append(batch)

In [39]:
before_sep = [parts[0] for parts in (string.split("[SEP]") for string in sensitive_batches)]
after_sep = [parts[1] for parts in (string.split("[SEP]") for string in sensitive_batches)]

device = torch.device("cpu")
cross_encoder_single = FullCrossEncoder(config).to(device)
bert_tokens = cross_encoder_single.tokenizer(text=before_sep,text_pair=after_sep,add_special_tokens=True,padding=True,truncation=True)

input_ids = torch.tensor(bert_tokens['input_ids'], device=device)
attention_mask = torch.tensor(bert_tokens['attention_mask'], device=device)

loading file vocab.json from cache at /home/sago/.cache/huggingface/hub/models--roberta-large/snapshots/716877d372b884cad6d419d828bac6c85b3b18d9/vocab.json
loading file merges.txt from cache at /home/sago/.cache/huggingface/hub/models--roberta-large/snapshots/716877d372b884cad6d419d828bac6c85b3b18d9/merges.txt
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at None
loading configuration file config.json from cache at /home/sago/.cache/huggingface/hub/models--roberta-large/snapshots/716877d372b884cad6d419d828bac6c85b3b18d9/config.json
Model config RobertaConfig {
  "_name_or_path": "roberta-large",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 

In [9]:
for instances,labels,instances_id,task_ids in train_dataloader:
    print(instances_id)
    print([extract_dataset_info_from_instance_id(instances_id) for instances_id in instances_id])

128

In [15]:
from collections import Counter

for instances,labels,instances_id,task_ids in train_dataloader:
    print(instances_id)
    print([extract_dataset_info_from_instance_id(instances_id) for instances_id in instances_id])
    break
    
    
    # dataset_names = []
    # for id in instances_id:
    #     parts = id.split("[SEP]", 1)
    #     sub_parts = parts[0].split("/")
    #     dataset_names.append(sub_parts[0])
    #     if 'how'==sub_parts[0]:
    #         print(sub_parts)
    #         break
    
    # value_counts = Counter(dataset_names)
    # output_strings = [f"{value}: {count}" for value, count in value_counts.items()]
    # print(', '.join(output_strings))



('mrpc/train[SEP]2806', 'mrpc/train[SEP]3419', 'mrpc/train[SEP]480', 'mrpc/train[SEP]2979', 'mrpc/train[SEP]3363', 'mrpc/train[SEP]2601', 'mrpc/train[SEP]3520', 'mrpc/train[SEP]2292', 'mrpc/train[SEP]3298', 'mrpc/train[SEP]3884', 'mrpc/train[SEP]2505', 'mrpc/train[SEP]1229', 'mrpc/train[SEP]2910', 'mrpc/train[SEP]947', 'mrpc/train[SEP]2561', 'mrpc/train[SEP]2660', 'mrpc/train[SEP]1661', 'mrpc/train[SEP]3291', 'mrpc/train[SEP]2902', 'mrpc/train[SEP]188', 'mrpc/train[SEP]2318', 'mrpc/train[SEP]3352', 'mrpc/train[SEP]3361', 'mrpc/train[SEP]3688', 'mrpc/train[SEP]2108', 'mrpc/train[SEP]3660', 'mrpc/train[SEP]2062', 'mrpc/train[SEP]2942', 'mrpc/train[SEP]17', 'mrpc/train[SEP]3637', 'mrpc/train[SEP]532', 'mrpc/train[SEP]4012')
[{'mention_type': 'NA', 'dataset_name': 'mrpc', 'split': 'train'}, {'mention_type': 'NA', 'dataset_name': 'mrpc', 'split': 'train'}, {'mention_type': 'NA', 'dataset_name': 'mrpc', 'split': 'train'}, {'mention_type': 'NA', 'dataset_name': 'mrpc', 'split': 'train'}, {'me

### Debug problematic mentions

In [10]:
# topics_list = set([mention["topic_id"] for mention in val.mentions])
# subtopics_list = set([mention["subtopic_id"] for mention in val.mentions])

# problematic_mentions = [mention for mention in train.mentions if min(mention['tokens_number_context'])>len(mention["mention_context"])]

# ids= []
# len_context = []
# min_mention_token_number = []
# for mention in problematic_mentions:
#     ids.append(mention["mention_id"])
#     len_context.append(len(mention["mention_context"]))
#     min_mention_token_number.append(min(mention['tokens_number_context']))

# import pandas as pd
# df = pd.DataFrame({"mention_id":ids,"context_length":len_context,"min_mention_token_number":min_mention_token_number})
# # Print the DataFrame in Markdown-friendly format
# markdown_table = df.to_markdown(index=False)
# print(len(df))
# print(markdown_table)

0
| mention_id   | context_length   | min_mention_token_number   |
|--------------|------------------|----------------------------|


## Create dataloaders, device and model

In [9]:
#Subset data in debug for dummy run
train_subset = data.Subset(train, range(8))
val_subset = data.Subset(val, range(8))

In [10]:
# sampler = MultiTaskBatchSampler(        dataset=train_subset,
#         batch_size=32,
#         mix_opt=1,
#         extra_task_ratio=1,
#         current_epoch=1,
#         total_epochs=10,
#         bin_size=64,
#         bin_on=False,
#         bin_grow_ratio=0.5,
#         sampling = "sequential")
        
# train_loader = DataLoader(
#         train_subset,
#         batch_sampler=sampler,
#         pin_memory=True
#         )
val_loader = data.DataLoader(train_subset, batch_size=config.batch_size, shuffle=True, pin_memory=True)
train_loader = data.DataLoader(val_subset, batch_size=config.batch_size, shuffle=False, pin_memory=True)

#device_ids = config.gpu_num
#device = torch.device("cuda:{}".format(device_ids[0]))
device = torch.device("cpu")

logger.info('Init models')
cross_encoder_single = FullCrossEncoder(config).to(device)
#cross_encoder = torch.nn.DataParallel(cross_encoder_single, device_ids=[device])
cross_encoder = cross_encoder_single

loading file vocab.json from cache at C:\Users\sago/.cache\huggingface\hub\models--roberta-large\snapshots\716877d372b884cad6d419d828bac6c85b3b18d9\vocab.json
loading file merges.txt from cache at C:\Users\sago/.cache\huggingface\hub\models--roberta-large\snapshots\716877d372b884cad6d419d828bac6c85b3b18d9\merges.txt
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at None
loading configuration file config.json from cache at C:\Users\sago/.cache\huggingface\hub\models--roberta-large\snapshots\716877d372b884cad6d419d828bac6c85b3b18d9\config.json
Model config RobertaConfig {
  "_name_or_path": "roberta-large",
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediat

In [11]:

# Define the names of the layers to freeze *empty to keep roberta frozen
unfreeze_layers = [
]
#Unfreezes Roberta layers in list
for name, param in cross_encoder_single.model.named_parameters():
    param.requires_grad = any(layer_name in name for layer_name in unfreeze_layers)

# Unfreeze the parameters of the linear layers
for param in cross_encoder_single.linear_cdcr.parameters():
    param.requires_grad = True

# Unfreeze the parameters of the linear layers
for param in cross_encoder_single.linear_pi.parameters():
    param.requires_grad = True


In [12]:
## Optimizer and loss function
criterion = get_loss_function(config)
optimizer = get_optimizer(config, [cross_encoder])
scheduler = get_scheduler(optimizer, total_steps=config.epochs * len(train_loader))



In [15]:
len(train_loader)

1

In [14]:
# model_params = list(cross_encoder.parameters())

# # with open('model_parameters.txt', 'w') as f:
# #     for name, param in cross_encoder.named_parameters():
# #         f.write(f"Layer: {name}, Parameters: {param.numel()}\n")

# for name, param in cross_encoder.named_parameters():
#     print(f"Layer: {name}, Parameters: {param.numel()}")

In [17]:
# # Revision prior to training
# cross_encoder.train()
# # Get one batch
# batch_x, batch_y, instance_id, dataset_task = next(iter(train_loader))

# before_sep = [parts[0] for parts in (string.split("[SEP]") for string in batch_x)]
# after_sep = [parts[1] for parts in (string.split("[SEP]") for string in batch_x)]

# bert_tokens = cross_encoder_single.tokenizer(text=before_sep,text_pair=after_sep,add_special_tokens=True,padding=True,truncation=True)
# cross_encoder_single.tokenizer.decode(bert_tokens['input_ids'][0])

In [63]:
##################################################################################
####                    TRAINING
##################################################################################

f1 = []
best_f1 = 0.0
update_interval = 10
train_results_per_epoch= {}
val_results_per_epoch= {}

for epoch in range(config.epochs):
    logger.info('Epoch: {}'.format(epoch))
    accumulate_loss = 0
    number_of_positive_pairs, number_of_pairs = 0, 0
    cross_encoder.train()
    all_train_scores, all_train_labels, all_train_identifiers = [], [], []

    for batch_x, batch_y, instance_id, dataset_task in tqdm(train_loader):
        #Separate sentences from batch to add the tokenizers special tokens
        before_sep = [parts[0] for parts in (string.split("[SEP]") for string in batch_x)]
        after_sep = [parts[1] for parts in (string.split("[SEP]") for string in batch_x)]
        all_train_identifiers.extend([extract_dataset_info_from_instance_id(id) for id in instance_id])

        bert_tokens = cross_encoder_single.tokenizer(text=before_sep,text_pair=after_sep,add_special_tokens=True,padding=True,truncation=True)
        input_ids = torch.tensor(bert_tokens['input_ids'], device=device)
        attention_mask = torch.tensor(bert_tokens['attention_mask'], device=device)

        optimizer.zero_grad()
        cross_encoder.zero_grad()
        scores = cross_encoder(input_ids, attention_mask,dataset_task)
        all_train_scores.extend(scores.squeeze(1))

        batch_y = batch_y.view(-1, 1)
        all_train_labels.extend(scores.squeeze(1))

        loss = criterion(scores, batch_y.to(device))
        loss.mean().backward()
        torch.nn.utils.clip_grad_norm_(cross_encoder.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        torch.cuda.empty_cache()


        accumulate_loss += loss.item()

        number_of_positive_pairs += len((batch_y == 1).nonzero())
        number_of_pairs += len(batch_y)

    # list_of_topics = shuffle(list(range(len(training_set.topic_list))))
    # number_of_positive_pairs = 0
    # number_of_pairs = 0
    # for topic_num in tqdm(list_of_topics):
    #
    #
    #
    #
    #     topic = training_set.topic_list[topic_num]
    #     first, second, labels = get_candidate_spans(training_set, topic)
    #
    #     number_of_positive_pairs += len((labels == 1).nonzero())
    #     number_of_pairs += len(labels)
    #
    #
    #     pairwises = CrossEncoderDataset(training_set.mentions_repr, first, second, labels)
    #     generator = data.DataLoader(pairwises, shuffle=True, batch_size=config.batch_size)
    #     loss = fine_tune_bert(cross_encoder, generator, criterion, optimizer, device)
    #     accumulate_loss += loss
    all_train_labels = torch.stack(all_train_labels).tolist()
    all_train_scores = torch.stack(all_train_scores).tolist()
    train_strict_preds = [int(score>0) for score in all_train_scores]
    train_results_per_epoch[epoch] = {'scores':train_strict_preds,'labels':all_train_labels,'identifiers':all_train_identifiers}

    logger.info('Number of positive/total pairs: {}/{}'.format(number_of_positive_pairs, number_of_pairs))
    logger.info('Accumulate loss: {}'.format(accumulate_loss))


    logger.info('Evaluate on the dev set')
    all_scores, all_labels,all_identifiers = [], [], []
    number_of_positive_pairs, number_of_pairs = 0, 0
    cross_encoder.eval()

    for batch_x, batch_y,instance_id,dataset_task in tqdm(val_loader):
        bert_tokens = cross_encoder_single.tokenizer(batch_x, pad_to_max_length=True)
        input_ids = torch.tensor(bert_tokens['input_ids'], device=device)
        attention_mask = torch.tensor(bert_tokens['attention_mask'], device=device)
        all_identifiers.extend([extract_dataset_info_from_instance_id(id) for id in instance_id])

        with torch.no_grad():
            scores = cross_encoder(input_ids, attention_mask,dataset_task)

        torch.cuda.empty_cache()
        
        number_of_positive_pairs += len((batch_y == 1).nonzero())
        number_of_pairs += len(batch_y)


    # for topic_num, topic in enumerate(tqdm(dev_set.topic_list)):
    #     dev_topic = dev_set.topic_list[topic_num]
    #     first, second, labels = get_candidate_spans(dev_set, dev_topic)
    #     number_of_positive_pairs += len((labels == 1).nonzero())
    #     number_of_pairs += len(labels)
    #
    #     pairwises = CrossEncoderDataset(dev_set.mentions_repr, first, second, labels)
    #     generator = data.DataLoader(pairwises, shuffle=True, batch_size=config.batch_size)
    #     scores, labels = evaluate_model(cross_encoder, generator, device)
    #     torch.cuda.empty_cache()
    #
        all_scores.extend(scores.squeeze())
        all_labels.extend(batch_y.squeeze())



    all_labels = torch.stack(all_labels)
    all_scores = torch.stack(all_scores)
    strict_preds = (all_scores > 0).to(torch.int)
    
    eval = Evaluation(strict_preds, all_labels.to(device))
    logger.info('Number of predictions: {}/{}'.format(strict_preds.sum(), len(strict_preds)))
    logger.info('Number of positive pairs: {}/{}'.format(len((all_labels == 1).nonzero()),
                                                            len(all_labels)))

    logger.info('Min score: {}'.format(all_scores.min().item()))
    logger.info('Max score: {}'.format(all_scores.max().item()))
    logger.info('Strict - Recall: {}, Precision: {}, F1: {}, Accuracy: {}'.
                format(eval.get_recall(), eval.get_precision(), eval.get_f1(), eval.get_accuracy()))

    val_results_per_epoch[epoch]= {'scores':strict_preds.tolist(),'labels':all_labels.tolist(),'identifiers':all_identifiers}
    
    out_dir = os.path.join(config.model_path, '{}_{}'.format(
        'base' if 'base' in config.roberta_model else 'large', config.batch_size),
        'checkpoint_{}'.format(epoch))
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    current_f1 = eval.get_f1() 
    if current_f1 > best_f1:
        best_f1 = current_f1
        cross_encoder_single.model.save_pretrained(os.path.join(out_dir, 'bert'))
        torch.save(cross_encoder_single.linear_cdcr.state_dict(), os.path.join(out_dir, 'linear_cdcr'))
        torch.save(cross_encoder_single.linear_cdcr.state_dict(), os.path.join(out_dir, 'linear_pi'))

    f1.append(eval.get_f1())
print("F1 values",f1)


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]C:\Users\sago\AppData\Roaming\Python\Python38\site-packages\transformers\tokenization_utils_base.py:2377: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
100%|██████████| 1/1 [00:28<00:00, 28.42s/it]
Configuration saved in models/pairwise_scorers\large_256\checkpoint_0\bert\config.json
Model weights saved in models/pairwise_scorers\large_256\checkpoint_0\bert\pytorch_model.bin
100%|██████████| 1/1 [00:30<00:00, 30.03s/it]

F1 values [1.0, 1.0, 1.0]


In [64]:
import json
# Specify the filename

filename_train = os.path.join(out_dir, "train_results.json")
filename_val = os.path.join(out_dir, "val_results.json")

# Write the JSON data to the file
with open(filename_train, 'w') as json_file:
    json.dump(train_results_per_epoch, json_file, indent=4)

# Write the JSON data to the file
with open(filename_val, 'w') as json_file:
    json.dump(val_results_per_epoch, json_file, indent=4)

{'scores': tensor([1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.int32),
 'labels': tensor([1., 1., 1., 1., 1., 1., 1., 1.]),
 'identifiers': [{'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'},
  {'mention_type': 'entity_mentions',
   'dataset_name': 'ECBplus-prep',
   'split': 'train'}]}

In [48]:
(torch.tensor(train_results_per_epoch[0]['scores'])> 0).to(torch.int)

tensor([1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.int32)

In [45]:
train_results_per_epoch[0]['scores']

TypeError: '>' not supported between instances of 'list' and 'int'